# Setup

In [1]:
from pathlib import Path
import sqlite3
import pickle

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "db" / "app.db"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH exists:", DB_PATH.exists(), DB_PATH)

PROJECT_ROOT: /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation
DB_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/app.db


In [2]:
# Load Documents from SQLite
conn = sqlite3.connect(DB_PATH)

docs_df = pd.read_sql_query("""
    SELECT
        id,
        source_doc_id,
        title,
        abstract,
        clean_text,
        publication_year,
        journal
    FROM documents
    ORDER BY id
""", conn)

conn.close()

print("Loaded documents shape:", docs_df.shape)
display(docs_df.head())

Loaded documents shape: (738, 7)


,id,source_doc_id,title,abstract,clean_text,publication_year,journal
0,1,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",Study protocol: Comparison of different risk p...,2020,Wellcome open research
1,2,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,"Mental health, sleep quality and quality of li...",2020,F1000Research
2,3,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,Parent perspectives on food allergy management...,2020,Journal of food allergy
3,4,38046874,COVID-19 critical illness in Sweden: character...,Objective: During the coronavirus disease 2019...,COVID-19 critical illness in Sweden: character...,2020,Critical care and resuscitation : journal of t...
4,5,38046867,Geolocated Twitter-based population mobility i...,"Using geotagged Twitter data in Victoria, we c...",Geolocated Twitter-based population mobility i...,2020,Critical care and resuscitation : journal of t...


In [3]:
# Check Embedding coverage
conn = sqlite3.connect(DB_PATH)

emb_count = pd.read_sql_query("""
    SELECT COUNT(*) AS n_embeddings
    FROM document_embeddings
""", conn)

doc_count = pd.read_sql_query("""
    SELECT COUNT(*) AS n_docs
    FROM documents
""", conn)

conn.close()

print("Documents:", doc_count["n_docs"].iloc[0])
print("Embeddings:", emb_count["n_embeddings"].iloc[0])

Documents: 738
Embeddings: 0


# 5-document Embedding Test

In [4]:
# Load embedding model
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Loaded model:", EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [5]:
# Embed a tiny sample
sample_texts = docs_df["clean_text"].head(5).tolist()

sample_embeddings = model.encode(
    sample_texts,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Embedding array shape:", sample_embeddings.shape)
print("Dtype:", sample_embeddings.dtype)
print("First 10 values of first embedding:")
print(sample_embeddings[0][:10])

Embedding array shape: (5, 384)
Dtype: float32
First 10 values of first embedding:
[-0.02217141 -0.04530515 -0.03541752  0.04755112  0.11915016  0.0387663
 -0.04033012  0.11091505 -0.02617893 -0.01154005]


In [6]:
# Helper to serialize embeddings
def serialize_embedding(vec: np.ndarray) -> bytes:
    return pickle.dumps(vec.astype(np.float32), protocol=pickle.HIGHEST_PROTOCOL)

In [7]:
# Prepare the insert test
sample_docs = docs_df.head(5).copy()

sample_embeddings = model.encode(
    sample_docs["clean_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=False
)

rows_to_insert = []
for doc_id, emb in zip(sample_docs["id"].tolist(), sample_embeddings):
    rows_to_insert.append((
        doc_id,
        serialize_embedding(emb),
        EMBEDDING_MODEL_NAME,
    ))

print("Rows prepared:", len(rows_to_insert))
print("Example document_id:", rows_to_insert[0][0])
print("Serialized bytes length:", len(rows_to_insert[0][1]))
print("Model name:", rows_to_insert[0][2])

Rows prepared: 5
Example document_id: 1
Serialized bytes length: 1664
Model name: all-MiniLM-L6-v2


In [8]:
# Insert 5 test embeddings
conn = sqlite3.connect(DB_PATH)

insert_sql = """
INSERT INTO document_embeddings (
    document_id,
    embedding,
    model_name
)
VALUES (?, ?, ?)
"""

before_n = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM document_embeddings",
    conn
)["n"].iloc[0]

conn.executemany(insert_sql, rows_to_insert)
conn.commit()

after_n = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM document_embeddings",
    conn
)["n"].iloc[0]

print("Embeddings before insert:", before_n)
print("Rows inserted:", len(rows_to_insert))
print("Embeddings after insert:", after_n)

conn.close()

Embeddings before insert: 0
Rows inserted: 5
Embeddings after insert: 5


In [9]:
# Readback check
conn = sqlite3.connect(DB_PATH)

check_df = pd.read_sql_query("""
    SELECT
        id,
        document_id,
        embedding,
        model_name
    FROM document_embeddings
    ORDER BY id
    LIMIT 1
""", conn)

conn.close()

row = check_df.iloc[0]
loaded_vec = pickle.loads(row["embedding"])

print("Embedding row id:", row["id"])
print("Document id:", row["document_id"])
print("Model name:", row["model_name"])
print("Loaded vector shape:", loaded_vec.shape)
print("Loaded vector dtype:", loaded_vec.dtype)
print("First 10 values:")
print(loaded_vec[:10])

Embedding row id: 1
Document id: 1
Model name: all-MiniLM-L6-v2
Loaded vector shape: (384,)
Loaded vector dtype: float32
First 10 values:
[-0.02217141 -0.04530515 -0.03541752  0.04755112  0.11915016  0.0387663
 -0.04033012  0.11091505 -0.02617893 -0.01154005]


In [10]:
# check current embedding coverage
conn = sqlite3.connect(DB_PATH)

coverage_df = pd.read_sql_query("""
    SELECT
        d.id AS document_id,
        d.title,
        CASE
            WHEN e.document_id IS NOT NULL THEN 1
            ELSE 0
        END AS has_embedding
    FROM documents d
    LEFT JOIN document_embeddings e
        ON d.id = e.document_id
       AND e.model_name = ?
    ORDER BY d.id
""", conn, params=[EMBEDDING_MODEL_NAME])

conn.close()

print("Coverage summary:")
print(coverage_df["has_embedding"].value_counts().sort_index())

display(coverage_df.head(10))

Coverage summary:
has_embedding
0    733
1      5
Name: count, dtype: int64


,document_id,title,has_embedding
0,1,Study protocol: Comparison of different risk p...,1
1,2,"Mental health, sleep quality and quality of li...",1
2,3,Parent perspectives on food allergy management...,1
3,4,COVID-19 critical illness in Sweden: character...,1
4,5,Geolocated Twitter-based population mobility i...,1
5,6,Public Involvement & Engagement in health ineq...,0
6,7,Using data linkage to monitor COVID-19 vaccina...,0
7,8,A living mapping review for COVID-19 funded re...,0
8,9,[COVID-19 and its socio-cultural imagery in La...,0
9,10,"[Risks, contamination and prevention against C...",0


# Scaling to all documents

In [11]:
# isolate documents missing embeddings
conn = sqlite3.connect(DB_PATH)

missing_df = pd.read_sql_query("""
    SELECT
        d.id,
        d.clean_text
    FROM documents d
    LEFT JOIN document_embeddings e
        ON d.id = e.document_id
       AND e.model_name = ?
    WHERE e.document_id IS NULL
    ORDER BY d.id
""", conn, params=[EMBEDDING_MODEL_NAME])

conn.close()

print("Documents missing embeddings:", len(missing_df))
display(missing_df.head())

Documents missing embeddings: 733


,id,clean_text
0,6,Public Involvement & Engagement in health ineq...
1,7,Using data linkage to monitor COVID-19 vaccina...
2,8,A living mapping review for COVID-19 funded re...
3,9,[COVID-19 and its socio-cultural imagery in La...
4,10,"[Risks, contamination and prevention against C..."


In [12]:
# batchs ettings for embedding
EMBED_BATCH_SIZE = 64

print("EMBED_BATCH_SIZE:", EMBED_BATCH_SIZE)

EMBED_BATCH_SIZE: 64


In [13]:
# embed missing docs in batches  and insert to SQL
conn = sqlite3.connect(DB_PATH)

insert_sql = """
INSERT INTO document_embeddings (
    document_id,
    embedding,
    model_name
)
VALUES (?, ?, ?)
"""

n_total = len(missing_df)
n_inserted = 0

for start in range(0, n_total, EMBED_BATCH_SIZE):
    end = min(start + EMBED_BATCH_SIZE, n_total)
    batch_df = missing_df.iloc[start:end]

    texts = batch_df["clean_text"].tolist()
    doc_ids = batch_df["id"].tolist()

    batch_embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    rows_to_insert = [
        (doc_id, serialize_embedding(emb), EMBEDDING_MODEL_NAME)
        for doc_id, emb in zip(doc_ids, batch_embeddings)
    ]

    conn.executemany(insert_sql, rows_to_insert)
    conn.commit()

    n_inserted += len(rows_to_insert)
    print(f"Inserted batch {start}-{end} | cumulative inserted: {n_inserted}/{n_total}")

conn.close()

Inserted batch 0-64 | cumulative inserted: 64/733
Inserted batch 64-128 | cumulative inserted: 128/733
Inserted batch 128-192 | cumulative inserted: 192/733
Inserted batch 192-256 | cumulative inserted: 256/733
Inserted batch 256-320 | cumulative inserted: 320/733
Inserted batch 320-384 | cumulative inserted: 384/733
Inserted batch 384-448 | cumulative inserted: 448/733
Inserted batch 448-512 | cumulative inserted: 512/733
Inserted batch 512-576 | cumulative inserted: 576/733
Inserted batch 576-640 | cumulative inserted: 640/733
Inserted batch 640-704 | cumulative inserted: 704/733
Inserted batch 704-733 | cumulative inserted: 733/733


In [14]:
# final embedding coverage check
conn = sqlite3.connect(DB_PATH)

coverage_summary = pd.read_sql_query("""
    SELECT
        COUNT(*) AS n_documents,
        SUM(CASE WHEN e.document_id IS NOT NULL THEN 1 ELSE 0 END) AS n_with_embeddings
    FROM documents d
    LEFT JOIN document_embeddings e
        ON d.id = e.document_id
       AND e.model_name = ?
""", conn, params=[EMBEDDING_MODEL_NAME])

emb_count = pd.read_sql_query("""
    SELECT COUNT(*) AS n_embeddings
    FROM document_embeddings
    WHERE model_name = ?
""", conn, params=[EMBEDDING_MODEL_NAME])

conn.close()

display(coverage_summary)
display(emb_count)

,n_documents,n_with_embeddings
0,738,738


,n_embeddings
0,738


In [15]:
# load a few embeddings from SQL
conn = sqlite3.connect(DB_PATH)

sample_emb_df = pd.read_sql_query("""
    SELECT
        e.document_id,
        e.embedding,
        e.model_name,
        d.title
    FROM document_embeddings e
    JOIN documents d
        ON e.document_id = d.id
    WHERE e.model_name = ?
    ORDER BY e.document_id
    LIMIT 5
""", conn, params=[EMBEDDING_MODEL_NAME])

conn.close()

loaded_embeddings = np.vstack([
    pickle.loads(blob) for blob in sample_emb_df["embedding"]
])

print("Loaded embedding matrix shape:", loaded_embeddings.shape)
print("Dtype:", loaded_embeddings.dtype)

display(sample_emb_df[["document_id", "model_name", "title"]])

Loaded embedding matrix shape: (5, 384)
Dtype: float32


,document_id,model_name,title
0,1,all-MiniLM-L6-v2,Study protocol: Comparison of different risk p...
1,2,all-MiniLM-L6-v2,"Mental health, sleep quality and quality of li..."
2,3,all-MiniLM-L6-v2,Parent perspectives on food allergy management...
3,4,all-MiniLM-L6-v2,COVID-19 critical illness in Sweden: character...
4,5,all-MiniLM-L6-v2,Geolocated Twitter-based population mobility i...


In [16]:
# similarity check
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(loaded_embeddings)

sim_df = pd.DataFrame(
    sim_matrix,
    index=sample_emb_df["document_id"],
    columns=sample_emb_df["document_id"]
)

print("Cosine similarity matrix:")
display(sim_df)

Cosine similarity matrix:


document_id,1,2,3,4,5
document_id,,,,,
1,1.000000,0.418934,0.483066,0.575339,0.473647
2,0.418934,1.000000,0.455046,0.427417,0.293598
3,0.483066,0.455046,1.000000,0.426654,0.282749
4,0.575339,0.427417,0.426654,1.000000,0.358371
5,0.473647,0.293598,0.282749,0.358371,1.000000


In [17]:
# TODO Add uniqueness constraint to prevent duplicate embeddings